# 🧠 SRS Ambiguity Detection — Fine-tune Qwen2.5-VL-3B

**Dataset:** 2187 SRS requirements, nhãn `ok` / `defect`  
**Model:** `Qwen/Qwen2.5-VL-3B-Instruct` (4-bit QLoRA)  
**Split:** 80% train / 10% val / 10% test  
**GPU target:** T4 15GB (Google Colab Free)

---
### Pipeline:
```
CSV Dataset → Prompt Template → QLoRA Fine-tune
                                      ↓
                          Epoch/Loss Visualization
                                      ↓
                          Back-testing on Test Set
                                      ↓
                    Metrics: Accuracy, F1, Precision, Recall,
                             Confusion Matrix, ROC-AUC
```
---
⚠️ **Trước khi chạy:** Runtime → Change runtime type → **T4 GPU**

## 📦 CELL 1 — Cài đặt thư viện

In [ ]:
# ============================================================
# CELL 1: Cài đặt — chạy 1 lần, restart runtime nếu được hỏi
# ============================================================

!pip install -q transformers>=4.45.0
!pip install -q peft>=0.12.0              # LoRA / QLoRA
!pip install -q trl>=0.11.0               # SFTTrainer
!pip install -q bitsandbytes>=0.43.0      # 4-bit quant
!pip install -q accelerate>=0.34.0
!pip install -q qwen-vl-utils             # Qwen2.5-VL utils
!pip install -q datasets
!pip install -q scikit-learn
!pip install -q matplotlib seaborn
!pip install -q pandas numpy

import os, warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print('✅ Cài đặt xong!')

## 📂 CELL 2 — Upload & kiểm tra dataset

In [ ]:
# ============================================================
# CELL 2: Upload dataset CSV
# File cần có 2 cột: 'requirement' và 'defect' (ok / defect)
# ============================================================

from google.colab import files
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print('📁 Upload file datasetRE.csv:')
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)
print(f'\n✅ Đã tải: {filename}')
print(f'   Tổng số mẫu  : {len(df)}')
print(f'   Cột          : {df.columns.tolist()}')
print(f'\nPhân phối nhãn:')
print(df['defect'].value_counts())

# --- Trực quan hóa phân phối ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
counts = df['defect'].value_counts()
axes[0].pie(counts.values, labels=counts.index,
            autopct='%1.1f%%', colors=['#4CAF50','#F44336'],
            startangle=90, textprops={'fontsize':13})
axes[0].set_title('Label Distribution', fontsize=14, fontweight='bold')

# Text length distribution
df['length'] = df['requirement'].str.len()
for label, color in [('ok','#4CAF50'), ('defect','#F44336')]:
    subset = df[df['defect'] == label]['length']
    axes[1].hist(subset, bins=40, alpha=0.6, label=label, color=color)
axes[1].set_xlabel('Số ký tự trong requirement', fontsize=12)
axes[1].set_ylabel('Số lượng', fontsize=12)
axes[1].set_title('Phân phối độ dài câu theo nhãn', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=12)

plt.tight_layout()
plt.savefig('dataset_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('\n📊 Biểu đồ dataset đã lưu: dataset_overview.png')

## ✂️ CELL 3 — Train/Val/Test Split (8-1-1)

In [ ]:
# ============================================================
# CELL 3: Stratified split 80/10/10
# Stratified = giữ nguyên tỉ lệ ok/defect trong mỗi split
# ============================================================

from sklearn.model_selection import train_test_split

# Bước 1: Tách test (10%)
df_trainval, df_test = train_test_split(
    df, test_size=0.10, random_state=42, stratify=df['defect']
)

# Bước 2: Tách val từ phần còn lại (10% / 90% ≈ 11.1%)
df_train, df_val = train_test_split(
    df_trainval, test_size=0.1111, random_state=42, stratify=df_trainval['defect']
)

df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print('📊 Kết quả split:')
print(f'   Train : {len(df_train):4d} mẫu  | ok={sum(df_train["defect"]=="ok")} | defect={sum(df_train["defect"]=="defect")}')
print(f'   Val   : {len(df_val):4d} mẫu  | ok={sum(df_val["defect"]=="ok")} | defect={sum(df_val["defect"]=="defect")}')
print(f'   Test  : {len(df_test):4d} mẫu  | ok={sum(df_test["defect"]=="ok")} | defect={sum(df_test["defect"]=="defect")}')
print(f'   Total : {len(df_train)+len(df_val)+len(df_test):4d} mẫu')

## 💬 CELL 4 — Prompt Template

In [ ]:
# ============================================================
# CELL 4: Định nghĩa prompt template cho fine-tuning
# Format: chat instruction → single-token output (ok/defect)
# ============================================================

SYSTEM_PROMPT = """You are an expert SRS (Software Requirements Specification) analyst.
Your task is to classify whether a given requirement contains a defect (ambiguity, vagueness, or incompleteness).
Respond with ONLY one word: either 'ok' or 'defect'. No explanation."""

def build_prompt(requirement: str, label: str = None) -> str:
    """
    Tạo prompt theo Qwen2.5 chat format.
    - Training: có label (answer)
    - Inference: không có label
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Requirement: {requirement.strip()}"}
    ]
    if label is not None:
        # Training format: bao gồm cả response để model học
        messages.append({"role": "assistant", "content": label})
    return messages

# --- Kiểm tra prompt ---
sample_req = df_train.iloc[0]['requirement']
sample_lbl = df_train.iloc[0]['defect']
print('📝 Ví dụ prompt (training):')
print(f'   Requirement : {sample_req[:80]}...')
print(f'   Label       : {sample_lbl}')
print()
for msg in build_prompt(sample_req, sample_lbl):
    print(f'[{msg["role"].upper()}] {msg["content"][:100]}')

## 🤖 CELL 5 — Load Qwen2.5-VL-3B với 4-bit QLoRA

In [ ]:
# ============================================================
# CELL 5: Load model từ HuggingFace với QLoRA config
# Download lần đầu ~7GB, sau đó cache lại
# ============================================================

import torch
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# --- Kiểm tra GPU ---
assert torch.cuda.is_available(), '❌ Không tìm thấy GPU! Hãy chọn T4 trong Runtime.'
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'🖥️  GPU  : {torch.cuda.get_device_name(0)}')
print(f'💾 VRAM : {vram_total:.1f} GB')

MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'

# --- 4-bit QLoRA quantization config ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f'\n📥 Đang tải model: {MODEL_ID}')
print('⏳ Lần đầu tải ~7GB, 3-5 phút...\n')

# --- Load Processor ---
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    padding_side='right'   # Quan trọng cho causal LM training
)

# Đảm bảo có pad token
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

# --- Load Model ---
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

# --- Chuẩn bị model cho k-bit training ---
model = prepare_model_for_kbit_training(model)

# --- LoRA config ---
# Target modules: các layer attention của Qwen2.5
lora_config = LoraConfig(
    r=16,                          # Rank: cân bằng giữa hiệu năng và VRAM
    lora_alpha=32,                 # Alpha = 2*r là thực hành chuẩn
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

# --- In thống kê tham số ---
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
vram_used = torch.cuda.memory_allocated() / 1e9

print('✅ Model đã sẵn sàng!')
print(f'   Trainable params : {trainable:,} ({100*trainable/total:.2f}%)')
print(f'   Total params     : {total:,}')
print(f'   VRAM đang dùng   : {vram_used:.2f} GB / {vram_total:.1f} GB')

## 🗃️ CELL 6 — Tạo HuggingFace Dataset

In [ ]:
# ============================================================
# CELL 6: Tokenize và tạo HuggingFace Dataset
# ============================================================

from datasets import Dataset

MAX_LENGTH = 256   # Đủ cho SRS requirements (avg 133 chars)

def tokenize_sample(example):
    """Tokenize một sample cho SFT training"""
    messages = build_prompt(example['requirement'], example['defect'])

    # Apply chat template
    text = processor.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    # Tokenize
    tokenized = processor.tokenizer(
        text,
        max_length=MAX_LENGTH,
        padding='max_length',
        truncation=True,
        return_tensors=None
    )

    # Labels = input_ids (causal LM: dự đoán token tiếp theo)
    # Mask padding tokens với -100 để không tính loss trên pad
    labels = tokenized['input_ids'].copy()
    labels = [
        -100 if token == processor.tokenizer.pad_token_id else token
        for token in labels
    ]
    tokenized['labels'] = labels

    return tokenized


# Tạo HuggingFace Datasets
print('🔄 Đang tokenize dataset...')

hf_train = Dataset.from_pandas(df_train[['requirement','defect']])
hf_val   = Dataset.from_pandas(df_val[['requirement','defect']])
hf_test  = Dataset.from_pandas(df_test[['requirement','defect']])

hf_train = hf_train.map(tokenize_sample, batched=False, desc='Tokenizing train')
hf_val   = hf_val.map(tokenize_sample,   batched=False, desc='Tokenizing val')

# Chỉ giữ các cột cần thiết
keep_cols = ['input_ids', 'attention_mask', 'labels']
hf_train = hf_train.select_columns(keep_cols)
hf_val   = hf_val.select_columns(keep_cols)

print(f'\n✅ Tokenize xong!')
print(f'   Train : {len(hf_train)} samples')
print(f'   Val   : {len(hf_val)} samples')
print(f'   Test  : {len(hf_test)} samples (tokenize lúc inference)')

## 🏋️ CELL 7 — Training với SFTTrainer

In [ ]:
# ============================================================
# CELL 7: Fine-tuning với SFTTrainer + logging đầy đủ
# Ước tính thời gian: ~20-40 phút với T4
# ============================================================

from transformers import TrainingArguments, DataCollatorForSeq2Seq, Trainer
import json

# --- Training Hyperparameters ---
EPOCHS        = 5      # Tăng nếu muốn train lâu hơn
BATCH_SIZE    = 4      # T4 safe với QLoRA + batch 4
GRAD_ACCUM    = 4      # Effective batch = 4*4 = 16
LR            = 2e-4   # Chuẩn cho QLoRA
WARMUP_RATIO  = 0.03
OUTPUT_DIR    = './srs_vlm_finetune'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type='cosine',
    fp16=True,                       # Tiết kiệm VRAM
    optim='paged_adamw_8bit',        # 8-bit optimizer cho T4
    logging_steps=10,                # Log mỗi 10 steps
    eval_strategy='epoch',           # Eval sau mỗi epoch
    save_strategy='epoch',
    save_total_limit=2,              # Giữ 2 checkpoint gần nhất
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',                # Tắt wandb
    dataloader_num_workers=0,
    remove_unused_columns=False,
)

# --- Data Collator ---
data_collator = DataCollatorForSeq2Seq(
    tokenizer=processor.tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8
)

# --- Custom Callback để lưu history ---
from transformers import TrainerCallback

class HistoryCallback(TrainerCallback):
    def __init__(self):
        self.train_loss_steps = []   # (step, loss)
        self.val_loss_epochs  = []   # (epoch, eval_loss)
        self.lr_history       = []   # (step, lr)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            if 'loss' in logs:
                self.train_loss_steps.append((state.global_step, logs['loss']))
            if 'eval_loss' in logs:
                self.val_loss_epochs.append((state.epoch, logs['eval_loss']))
            if 'learning_rate' in logs:
                self.lr_history.append((state.global_step, logs['learning_rate']))

history_cb = HistoryCallback()

# --- Trainer ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_val,
    data_collator=data_collator,
    callbacks=[history_cb],
)

print('🚀 Bắt đầu training...')
print(f'   Epochs         : {EPOCHS}')
print(f'   Batch size     : {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM} (effective)')
print(f'   Learning rate  : {LR}')
print(f'   Train samples  : {len(hf_train)}')
print(f'   Steps/epoch    : {len(hf_train) // (BATCH_SIZE * GRAD_ACCUM)}')
print()

train_result = trainer.train()

print('\n✅ Training xong!')
print(f'   Training loss cuối : {train_result.training_loss:.4f}')
print(f'   Thời gian train    : {train_result.metrics["train_runtime"]:.1f} giây')

# Lưu history để dùng ở các cell sau
with open('training_history.json', 'w') as f:
    json.dump({
        'train_loss_steps': history_cb.train_loss_steps,
        'val_loss_epochs':  history_cb.val_loss_epochs,
        'lr_history':       history_cb.lr_history
    }, f)
print('   History lưu tại   : training_history.json')

## 📈 CELL 8 — Trực quan hóa Training

In [ ]:
# ============================================================
# CELL 8: Visualize epoch / loss / learning rate
# ============================================================

import json
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# Load history
with open('training_history.json') as f:
    hist = json.load(f)

train_steps  = [x[0] for x in hist['train_loss_steps']]
train_losses = [x[1] for x in hist['train_loss_steps']]
val_epochs   = [x[0] for x in hist['val_loss_epochs']]
val_losses   = [x[1] for x in hist['val_loss_epochs']]
lr_steps     = [x[0] for x in hist['lr_history']]
lr_values    = [x[1] for x in hist['lr_history']]

# --- Smoothing train loss cho dễ nhìn ---
def smooth(values, weight=0.8):
    smoothed = []
    last = values[0]
    for v in values:
        last = last * weight + v * (1 - weight)
        smoothed.append(last)
    return smoothed

train_smooth = smooth(train_losses)

# --- Figure ---
fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 2, hspace=0.4, wspace=0.35)

COLORS = {'train':'#2196F3', 'val':'#F44336', 'lr':'#FF9800', 'smooth':'#1565C0'}

# Plot 1: Training Loss (step-level)
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(train_steps, train_losses, alpha=0.3, color=COLORS['train'], linewidth=1, label='Raw')
ax1.plot(train_steps, train_smooth, color=COLORS['smooth'], linewidth=2.5, label='Smoothed (β=0.8)')
ax1.set_xlabel('Training Step', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training Loss (Step-Level)', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Plot 2: Val Loss theo epoch
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(val_epochs, val_losses, 'o-', color=COLORS['val'],
         linewidth=2.5, markersize=8, markerfacecolor='white', markeredgewidth=2)
best_idx = int(np.argmin(val_losses))
ax2.axvline(x=val_epochs[best_idx], color='gray', linestyle='--', alpha=0.6)
ax2.annotate(f'Best\n{val_losses[best_idx]:.4f}',
             xy=(val_epochs[best_idx], val_losses[best_idx]),
             xytext=(val_epochs[best_idx]+0.1, val_losses[best_idx]+0.002),
             fontsize=10, color='gray')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Validation Loss', fontsize=12)
ax2.set_title('Validation Loss (Epoch-Level)', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Plot 3: Train vs Val Loss so sánh
ax3 = fig.add_subplot(gs[1, 0])
# Map train loss sang epoch
steps_per_epoch = max(train_steps) / len(val_epochs) if val_epochs else 1
train_epochs_x  = [s / steps_per_epoch for s in train_steps]
ax3.plot(train_epochs_x, train_smooth, color=COLORS['train'], linewidth=2, label='Train (smoothed)')
ax3.plot(val_epochs, val_losses, 'o-', color=COLORS['val'],
         linewidth=2.5, markersize=8, label='Validation')
ax3.set_xlabel('Epoch', fontsize=12)
ax3.set_ylabel('Loss', fontsize=12)
ax3.set_title('Train vs Validation Loss', fontsize=13, fontweight='bold')
ax3.legend(fontsize=11)
ax3.grid(True, alpha=0.3)

# Chú thích overfitting nếu có
if len(val_losses) >= 3:
    last3_val = val_losses[-3:]
    if last3_val[-1] > last3_val[0]:
        ax3.text(0.98, 0.95, '⚠️ Possible overfitting',
                 transform=ax3.transAxes, ha='right', va='top',
                 fontsize=10, color='red',
                 bbox=dict(boxstyle='round', facecolor='#FFEBEE', alpha=0.8))

# Plot 4: Learning Rate Schedule
ax4 = fig.add_subplot(gs[1, 1])
if lr_steps:
    ax4.plot(lr_steps, lr_values, color=COLORS['lr'], linewidth=2)
    ax4.fill_between(lr_steps, lr_values, alpha=0.15, color=COLORS['lr'])
ax4.set_xlabel('Training Step', fontsize=12)
ax4.set_ylabel('Learning Rate', fontsize=12)
ax4.set_title('Learning Rate Schedule (Cosine Warmup)', fontsize=13, fontweight='bold')
ax4.ticklabel_format(axis='y', style='sci', scilimits=(0,0))
ax4.grid(True, alpha=0.3)

fig.suptitle('SRS Ambiguity Detection — Training Dashboard', fontsize=16, fontweight='bold', y=1.01)
plt.savefig('training_dashboard.png', dpi=130, bbox_inches='tight')
plt.show()
print('📊 Dashboard lưu tại: training_dashboard.png')

# --- Summary table ---
print('\n📋 Training Summary:')
print(f'   Initial train loss : {train_losses[0]:.4f}')
print(f'   Final train loss   : {train_losses[-1]:.4f}')
print(f'   Best val loss      : {min(val_losses):.4f}  (epoch {val_epochs[best_idx]})')
print(f'   Final val loss     : {val_losses[-1]:.4f}')

## 🔮 CELL 9 — Back-testing trên Test Set

In [ ]:
# ============================================================
# CELL 9: Inference trên toàn bộ test set
# Ước tính: ~1-3 giây/sample → vài phút tổng
# ============================================================

import torch
from tqdm import tqdm

model.eval()

def predict(requirement: str) -> str:
    """
    Predict nhãn cho 1 requirement.
    Returns: 'ok' hoặc 'defect'
    """
    messages = build_prompt(requirement)  # Không có label

    text_input = processor.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor.tokenizer(
        text_input,
        return_tensors='pt',
        truncation=True,
        max_length=256
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=5,    # Chỉ cần 1 token (ok/defect)
            do_sample=False,
            temperature=1.0,
            pad_token_id=processor.tokenizer.pad_token_id,
        )

    # Decode chỉ phần generated (không decode prompt)
    new_tokens = output_ids[0, inputs['input_ids'].shape[1]:]
    prediction = processor.tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()

    # Normalize: nếu không phải ok/defect thì fallback
    if 'defect' in prediction:
        return 'defect'
    elif 'ok' in prediction:
        return 'ok'
    else:
        return 'ok'  # fallback về majority class


# --- Chạy inference trên toàn test set ---
print(f'🔮 Back-testing trên {len(df_test)} test samples...')

y_true = df_test['defect'].tolist()
y_pred = []

for req in tqdm(df_test['requirement'], desc='Predicting'):
    pred = predict(req)
    y_pred.append(pred)

df_test['predicted'] = y_pred
df_test['correct']   = df_test['defect'] == df_test['predicted']

print(f'\n✅ Inference xong!')
print(f'   Accuracy quick check: {sum(df_test["correct"]) / len(df_test):.4f}')

## 📊 CELL 10 — Evaluation Metrics & Visualization

In [ ]:
# ============================================================
# CELL 10: Full evaluation metrics + visualization
# Metrics: Accuracy, Precision, Recall, F1, MCC, ROC-AUC
# Plots: Confusion Matrix, ROC, Per-class bar chart
# ============================================================

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, matthews_corrcoef
)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import numpy as np

# Binary encoding cho ROC
label2int = {'ok': 0, 'defect': 1}
y_true_bin = [label2int[l] for l in y_true]
y_pred_bin = [label2int[l] for l in y_pred]

# --- Tính metrics ---
acc       = accuracy_score(y_true_bin, y_pred_bin)
prec_d    = precision_score(y_true_bin, y_pred_bin, pos_label=1)
rec_d     = recall_score(y_true_bin,    y_pred_bin, pos_label=1)
f1_d      = f1_score(y_true_bin,        y_pred_bin, pos_label=1)
prec_ok   = precision_score(y_true_bin, y_pred_bin, pos_label=0)
rec_ok    = recall_score(y_true_bin,    y_pred_bin, pos_label=0)
f1_ok     = f1_score(y_true_bin,        y_pred_bin, pos_label=0)
f1_macro  = f1_score(y_true_bin,        y_pred_bin, average='macro')
f1_weight = f1_score(y_true_bin,        y_pred_bin, average='weighted')
mcc       = matthews_corrcoef(y_true_bin, y_pred_bin)
cm        = confusion_matrix(y_true_bin, y_pred_bin)

# ROC: cần probability, dùng proxy (pred==defect → 1.0, 0.0)
y_score   = [1.0 if p == 1 else 0.0 for p in y_pred_bin]
try:
    auc = roc_auc_score(y_true_bin, y_score)
    fpr, tpr, _ = roc_curve(y_true_bin, y_score)
except:
    auc = float('nan')

# --- Print report ---
print('=' * 55)
print('         EVALUATION REPORT — TEST SET')
print('=' * 55)
print(f'  Accuracy         : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  F1 (macro)       : {f1_macro:.4f}')
print(f'  F1 (weighted)    : {f1_weight:.4f}')
print(f'  MCC              : {mcc:.4f}  (range: -1 ~ 1)')
print(f'  ROC-AUC          : {auc:.4f}')
print()
print(f'  Class: defect')
print(f'    Precision      : {prec_d:.4f}')
print(f'    Recall         : {rec_d:.4f}')
print(f'    F1-score       : {f1_d:.4f}')
print()
print(f'  Class: ok')
print(f'    Precision      : {prec_ok:.4f}')
print(f'    Recall         : {rec_ok:.4f}')
print(f'    F1-score       : {f1_ok:.4f}')
print('=' * 55)
print()
print('Confusion Matrix (rows=actual, cols=predicted):')
print(f'              ok    defect')
print(f'  actual ok  : {cm[0][0]:4d}    {cm[0][1]:4d}')
print(f'  actual def : {cm[1][0]:4d}    {cm[1][1]:4d}')

# --------------------------------------------------------
# VISUALIZATION
# --------------------------------------------------------
fig = plt.figure(figsize=(18, 13))
gs  = gridspec.GridSpec(2, 3, hspace=0.45, wspace=0.38)

# --- Plot 1: Confusion Matrix (Normalized) ---
ax1 = fig.add_subplot(gs[0, 0])
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(
    cm_norm, annot=True, fmt='.2%', cmap='Blues',
    xticklabels=['ok','defect'], yticklabels=['ok','defect'],
    ax=ax1, linewidths=1, cbar_kws={'label': 'Rate'},
    annot_kws={'size': 14, 'weight': 'bold'}
)
ax1.set_xlabel('Predicted', fontsize=12, fontweight='bold')
ax1.set_ylabel('Actual', fontsize=12, fontweight='bold')
ax1.set_title('Confusion Matrix (Normalized)', fontsize=13, fontweight='bold')

# --- Plot 2: Confusion Matrix (Raw counts) ---
ax2 = fig.add_subplot(gs[0, 1])
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Oranges',
    xticklabels=['ok','defect'], yticklabels=['ok','defect'],
    ax=ax2, linewidths=1,
    annot_kws={'size': 16, 'weight': 'bold'}
)
ax2.set_xlabel('Predicted', fontsize=12, fontweight='bold')
ax2.set_ylabel('Actual', fontsize=12, fontweight='bold')
ax2.set_title('Confusion Matrix (Raw Counts)', fontsize=13, fontweight='bold')

# --- Plot 3: Per-class Metrics Bar ---
ax3 = fig.add_subplot(gs[0, 2])
metrics_names  = ['Precision', 'Recall', 'F1-Score']
ok_scores      = [prec_ok, rec_ok, f1_ok]
defect_scores  = [prec_d,  rec_d,  f1_d]
x = np.arange(len(metrics_names))
w = 0.35
b1 = ax3.bar(x - w/2, ok_scores,     w, label='ok',     color='#4CAF50', alpha=0.85, edgecolor='white')
b2 = ax3.bar(x + w/2, defect_scores, w, label='defect', color='#F44336', alpha=0.85, edgecolor='white')
for bar in list(b1) + list(b2):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
ax3.set_xticks(x)
ax3.set_xticklabels(metrics_names, fontsize=12)
ax3.set_ylim(0, 1.12)
ax3.set_ylabel('Score', fontsize=12)
ax3.set_title('Per-Class Metrics', fontsize=13, fontweight='bold')
ax3.legend(fontsize=11)
ax3.grid(axis='y', alpha=0.3)

# --- Plot 4: ROC Curve ---
ax4 = fig.add_subplot(gs[1, 0])
if not np.isnan(auc):
    ax4.plot(fpr, tpr, color='#2196F3', lw=2.5, label=f'AUC = {auc:.4f}')
ax4.plot([0,1], [0,1], 'k--', lw=1.5, alpha=0.5, label='Random')
ax4.fill_between(fpr if not np.isnan(auc) else [0,1],
                  tpr if not np.isnan(auc) else [0,1], alpha=0.08, color='#2196F3')
ax4.set_xlabel('False Positive Rate', fontsize=12)
ax4.set_ylabel('True Positive Rate', fontsize=12)
ax4.set_title('ROC Curve', fontsize=13, fontweight='bold')
ax4.legend(fontsize=12, loc='lower right')
ax4.grid(True, alpha=0.3)

# --- Plot 5: Overall Metrics Summary ---
ax5 = fig.add_subplot(gs[1, 1])
summary_names  = ['Accuracy', 'F1 Macro', 'F1 Weighted', 'MCC\n(norm)', 'ROC-AUC']
mcc_norm = (mcc + 1) / 2  # Normalize -1~1 → 0~1
summary_scores = [acc, f1_macro, f1_weight, mcc_norm, auc if not np.isnan(auc) else 0]
colors_bar = ['#2196F3','#9C27B0','#673AB7','#009688','#FF5722']
bars = ax5.bar(summary_names, summary_scores, color=colors_bar, alpha=0.85, edgecolor='white', width=0.6)
for bar, score in zip(bars, summary_scores):
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{score:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax5.set_ylim(0, 1.15)
ax5.set_ylabel('Score', fontsize=12)
ax5.set_title('Overall Metrics Summary', fontsize=13, fontweight='bold')
ax5.axhline(y=0.8, color='gray', linestyle='--', alpha=0.4, linewidth=1)
ax5.text(4.5, 0.81, '0.8 threshold', fontsize=8, color='gray', ha='right')
ax5.grid(axis='y', alpha=0.3)

# --- Plot 6: Prediction Distribution ---
ax6 = fig.add_subplot(gs[1, 2])
pred_counts  = pd.Series(y_pred).value_counts()
true_counts  = pd.Series(y_true).value_counts()
labels_all   = ['ok', 'defect']
x = np.arange(len(labels_all))
true_vals = [true_counts.get(l, 0) for l in labels_all]
pred_vals = [pred_counts.get(l, 0) for l in labels_all]
b1 = ax6.bar(x - 0.2, true_vals, 0.4, label='Actual',    color='#607D8B', alpha=0.8)
b2 = ax6.bar(x + 0.2, pred_vals, 0.4, label='Predicted', color='#FF9800', alpha=0.8)
for bar in list(b1) + list(b2):
    ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)
ax6.set_xticks(x)
ax6.set_xticklabels(labels_all, fontsize=12)
ax6.set_ylabel('Count', fontsize=12)
ax6.set_title('Actual vs Predicted Distribution', fontsize=13, fontweight='bold')
ax6.legend(fontsize=11)
ax6.grid(axis='y', alpha=0.3)

fig.suptitle('SRS Ambiguity Detection — Evaluation Dashboard', fontsize=16, fontweight='bold', y=1.01)
plt.savefig('evaluation_dashboard.png', dpi=130, bbox_inches='tight')
plt.show()
print('📊 Evaluation dashboard lưu tại: evaluation_dashboard.png')

## 🔍 CELL 11 — Phân tích lỗi (Error Analysis)

In [ ]:
# ============================================================
# CELL 11: Phân tích các trường hợp mô hình dự đoán sai
# ============================================================

import textwrap

errors = df_test[~df_test['correct']].copy()
fn = errors[(errors['defect'] == 'defect') & (errors['predicted'] == 'ok')]   # False Negative
fp = errors[(errors['defect'] == 'ok')     & (errors['predicted'] == 'defect')]  # False Positive

print('=' * 60)
print('         ERROR ANALYSIS')
print('=' * 60)
print(f'Tổng lỗi        : {len(errors)} / {len(df_test)} ({len(errors)/len(df_test)*100:.1f}%)')
print(f'False Negatives : {len(fn)}  (defect bị bỏ sót → nguy hiểm hơn)')
print(f'False Positives : {len(fp)}  (ok bị đánh nhầm là defect)')

print('\n--- 5 False Negatives (defect nhưng predict là ok) ---')
for i, (_, row) in enumerate(fn.head(5).iterrows()):
    print(f'\n[FN {i+1}] {textwrap.fill(row["requirement"], 70)}')

print('\n--- 5 False Positives (ok nhưng predict là defect) ---')
for i, (_, row) in enumerate(fp.head(5).iterrows()):
    print(f'\n[FP {i+1}] {textwrap.fill(row["requirement"], 70)}')

# Lưu errors ra CSV
errors[['requirement','defect','predicted']].to_csv('error_analysis.csv', index=False)
print('\n💾 Lưu errors ra: error_analysis.csv')

## 💾 CELL 12 — Lưu model & xuất kết quả

In [ ]:
# ============================================================
# CELL 12: Lưu LoRA adapter và kết quả test ra CSV
# ============================================================

import json
from google.colab import files

# --- Lưu LoRA adapter ---
SAVE_PATH = './srs_lora_adapter'
model.save_pretrained(SAVE_PATH)
processor.tokenizer.save_pretrained(SAVE_PATH)
print(f'✅ LoRA adapter lưu tại: {SAVE_PATH}/')

# --- Lưu metrics ---
metrics_dict = {
    'test_samples'    : len(df_test),
    'accuracy'        : round(acc, 4),
    'f1_macro'        : round(f1_macro, 4),
    'f1_weighted'     : round(f1_weight, 4),
    'mcc'             : round(mcc, 4),
    'roc_auc'         : round(float(auc), 4) if not np.isnan(auc) else None,
    'defect_precision': round(prec_d, 4),
    'defect_recall'   : round(rec_d, 4),
    'defect_f1'       : round(f1_d, 4),
    'ok_precision'    : round(prec_ok, 4),
    'ok_recall'       : round(rec_ok, 4),
    'ok_f1'           : round(f1_ok, 4),
    'confusion_matrix': cm.tolist()
}
with open('evaluation_metrics.json', 'w') as f:
    json.dump(metrics_dict, f, indent=2)

# --- Xuất kết quả predictions ra CSV ---
df_test[['requirement','defect','predicted','correct']].to_csv(
    'test_predictions.csv', index=False
)

print('\n📁 Các file đã lưu:')
print('   evaluation_metrics.json  — tất cả metrics')
print('   test_predictions.csv     — prediction trên test set')
print('   error_analysis.csv       — các sample bị dự đoán sai')
print('   training_dashboard.png   — biểu đồ training')
print('   evaluation_dashboard.png — biểu đồ evaluation')

# --- Download ---
print('\n📥 Đang download kết quả...')
for fname in ['evaluation_metrics.json', 'test_predictions.csv',
              'training_dashboard.png', 'evaluation_dashboard.png']:
    files.download(fname)

print('\n✅ Hoàn tất!')

## 🧪 CELL 13 — Demo: Test câu bất kỳ

In [ ]:
# ============================================================
# CELL 13: Demo inference — nhập câu SRS bất kỳ
# ============================================================

TEST_CASES = [
    # Câu có defect
    "The system shall handle multiple sessions quickly and provide adequate performance.",
    "All data should be validated before it is stored.",
    # Câu ok
    "The system shall respond to all API requests within 200 milliseconds under normal load.",
    "The login module shall authenticate users using SHA-256 hashed passwords stored in PostgreSQL.",
    # Câu của bạn
    # "Your SRS requirement here...",
]

print('🧪 Demo Inference\n')
print('─' * 65)

for i, req in enumerate(TEST_CASES, 1):
    pred = predict(req)
    icon = '🔴' if pred == 'defect' else '✅'
    print(f'[{i}] {icon} {pred.upper()}')
    print(f'     {req[:75]}...' if len(req) > 75 else f'     {req}')
    print()